# 01 (run_02) — generator-ready splits for the 20k run

Changes vs. run_01 version:
- Paths point to `run_02`; PLOS threshold = lab rule (mean + 2*std) computed on all stays (13.99 d).
- Adds `encounter_type` and `encounter_reason` (top 20 reasons, rest = "other") from raw `encounters.csv`.
- Modes are defined from the encounter reason (5 modes, 98.7% of PLOS stays covered) and stored as binary
  `mode_*` columns, so notebooks 02–04 work unchanged (meta["modes"][k] = ["mode_k"]).
- Patient-level 70/15/15 split, seed 42.


In [ ]:
# Cell 1 — paths and settings
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/genai_synthea/run_02"
DATA = f"{BASE}/processed/Dataset_Tabular.csv"
RAW  = f"{BASE}/raw/csv"
OUT  = f"{BASE}/genai"
SEED = 42
TOP_REASONS = 20


In [ ]:
# Cell 2 — load, attach encounter type/reason
import os, json, numpy as np, pandas as pd
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(DATA, low_memory=False)
enc = pd.read_csv(f"{RAW}/encounters.csv", usecols=["Id", "ENCOUNTERCLASS", "DESCRIPTION", "REASONDESCRIPTION"])
enc = enc[enc["ENCOUNTERCLASS"] == "inpatient"].rename(columns={
    "Id": "hadm_id", "DESCRIPTION": "encounter_type", "REASONDESCRIPTION": "encounter_reason"})
df = df.merge(enc[["hadm_id", "encounter_type", "encounter_reason"]], on="hadm_id", how="left")
df["target"] = pd.to_numeric(df["target"], errors="coerce")
df = df.dropna(subset=["target"]).copy()
PLOS_THRESHOLD = float(df["target"].mean() + 2 * df["target"].std())
df["is_plos"] = (df["target"] > PLOS_THRESHOLD).astype(int)
print(f"Rows {len(df):,} | patients {df['subject_id'].nunique():,} | threshold {PLOS_THRESHOLD:.2f} d | PLOS {df['is_plos'].sum()} ({df['is_plos'].mean():.1%})")


In [ ]:
# Cell 3 — reason-based modes as binary columns
REASON_MODES = {
    "addiction":       ["Dependent drug abuse"],
    "lung_cancer":     ["Non-small cell carcinoma of lung", "small cell malignant neoplasm of lung"],
    "cardiopulmonary": ["congestive heart failure", "obstructive bronchitis", "emphysema"],
    "onc_transplant":  ["transplantation", "leukemia", "neoplasm of colon"],
    "infectious":      ["coronavirus"],
}
r = df["encounter_reason"].fillna("").str.lower()
MODE_COLS = []
for k, kws in REASON_MODES.items():
    col = f"mode_{k}"
    df[col] = r.apply(lambda s: any(w.lower() in s for w in kws)).astype("int8")
    MODE_COLS.append(col)
unexpl = (df[MODE_COLS].sum(axis=1) == 0)
print(pd.DataFrame({"plos_n": df.loc[df.is_plos == 1, MODE_COLS].sum(),
                    "plos_share": df.loc[df.is_plos == 1, MODE_COLS].mean()}).round(3))
print(f"Unexplained PLOS: {unexpl[df.is_plos == 1].mean():.1%}")


In [ ]:
# Cell 4 — features
TEXT_COLS = ["hpi", "social_history", "chief_complaint", "vitals", "labs", "conditions", "medications"]
df = df.drop(columns=[c for c in TEXT_COLS + ["birthdate"] if c in df.columns])
ID_COLS  = ["subject_id", "hadm_id"]
NUM_COLS = ["anchor_age"]
top = df["encounter_reason"].value_counts().head(TOP_REASONS).index
df["encounter_reason"] = df["encounter_reason"].where(df["encounter_reason"].isin(top), "other").fillna("other")
df["encounter_type"] = df["encounter_type"].fillna("unknown")
CAT_COLS = [c for c in ["gender", "race", "ethnicity", "admission_type", "encounter_type", "encounter_reason"] if c in df.columns]
for c in CAT_COLS: df[c] = df[c].astype(str)
COND_COLS = [c for c in df.columns if c.startswith("snomed_")]
df[COND_COLS] = df[COND_COLS].fillna(0).astype("int8")
BIN_COLS = COND_COLS + MODE_COLS
df["anchor_age"] = pd.to_numeric(df["anchor_age"], errors="coerce")
print(f"{len(NUM_COLS)} numeric | {len(CAT_COLS)} categorical {CAT_COLS} | {len(COND_COLS)} condition flags + {len(MODE_COLS)} mode flags")


In [ ]:
# Cell 5 — patient-level split 70/15/15
rng = np.random.default_rng(SEED)
patients = rng.permutation(np.array(sorted(df["subject_id"].astype(str).unique()), dtype=object))
n = len(patients)
tr, va = set(patients[:int(.70 * n)]), set(patients[int(.70 * n):int(.85 * n)])
split = df["subject_id"].astype(str).map(lambda s: "train" if s in tr else ("val" if s in va else "test"))
train, val, test = (df[split == s].copy() for s in ["train", "val", "test"])
for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {d['subject_id'].nunique():5d} patients | {len(d):6d} stays | PLOS {d['is_plos'].sum():4d} ({d['is_plos'].mean():.1%}) | "
          f"rare modes in PLOS: infectious {int(d.loc[d.is_plos==1,'mode_infectious'].sum())}, onc_transplant {int(d.loc[d.is_plos==1,'mode_onc_transplant'].sum())}")


In [ ]:
# Cell 6 — drop columns with no information in TRAIN
const_cat = [c for c in CAT_COLS if train[c].nunique() <= 1]
zero_bin  = [c for c in COND_COLS if train[c].sum() == 0]
for d in (train, val, test): d.drop(columns=const_cat + zero_bin, inplace=True)
CAT_COLS  = [c for c in CAT_COLS if c not in const_cat]
COND_COLS = [c for c in COND_COLS if c not in zero_bin]
BIN_COLS  = COND_COLS + MODE_COLS
print(f"Dropped constant: {const_cat} | all-zero condition flags: {len(zero_bin)}")


In [ ]:
# Cell 7 — save splits + meta (same contract as run_01, so 02-04 run unchanged)
cols = ID_COLS + NUM_COLS + CAT_COLS + BIN_COLS + ["target", "is_plos"]
for name, d in [("train", train), ("val", val), ("test", test)]:
    d[cols].to_parquet(f"{OUT}/{name}.parquet", index=False)
meta = {"source": DATA, "run": "run_02", "plos_threshold": PLOS_THRESHOLD, "seed": SEED,
        "id_cols": ID_COLS, "num_cols": NUM_COLS, "cat_cols": CAT_COLS, "bin_cols": BIN_COLS,
        "cond_cols": COND_COLS, "target": "target", "label": "is_plos",
        "modes": {c.replace("mode_", ""): [c] for c in MODE_COLS}, "reason_modes": REASON_MODES,
        "sizes": {k: {"stays": int(len(d)), "patients": int(d["subject_id"].nunique()), "plos": int(d["is_plos"].sum())}
                  for k, d in [("train", train), ("val", val), ("test", test)]}}
json.dump(meta, open(f"{OUT}/meta.json", "w"), indent=2)
print("Saved to", OUT); print(json.dumps(meta["sizes"], indent=2))
